## TITANIC

In [17]:
import os #path handling
import numpy as np #import numpy drives sklearn to use numpy arrays instead of python lists
import pandas as pd #CSV and dataframe handling
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier #kNN classifier
from sklearn.model_selection import train_test_split #Data set splitting functions
from sklearn.metrics import confusion_matrix #Confusion matrix
#Needed for SGD
from sklearn.linear_model import SGDClassifier 
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
#Needed for linear classification
from sklearn.svm import LinearSVC
#Needed for neural network
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report


In [18]:
dataPath = './' # path to folder containing the Iris data
dataFile = os.path.join(dataPath,'titanic.csv') # data file to use

In [19]:
# we assign column names based on the description file. 'FType' stands for 'Flower Type'
fullDF = pd.read_csv(dataFile,header=None,names=['Age','Cabin','Embarked','Fare','Name','Parch','Passengerld','Pclass','Sex','SibSP','Survived','Ticket']) 
fullDF.sample(10) # let's take a random sample from the full data frame


,Age,Cabin,Embarked,Fare,Name,Parch,Passengerld,Pclass,Sex,SibSP,Survived,Ticket
1166,19.0,D30,S,53.1000,"Marvin, Mr. Daniel Warner",0,749,1,male,1,0,113773
659,NaN,NaN,Q,15.5000,"Murphy, Miss. Katherine ""Kate""",0,242,3,female,1,1,367230
157,23.0,NaN,S,7.8542,"Lundin, Miss. Olga Elida",0,1049,3,female,0,1,347469
219,NaN,NaN,S,8.0500,"Thomson, Mr. Alexander Morrison",0,1111,3,male,0,0,32302
1123,39.0,NaN,S,26.0000,"Morley, Mr. Henry Samuel (""Mr Henry Marshall"")",0,706,2,male,0,0,250655
1222,27.0,NaN,S,6.9750,"Hedman, Mr. Oskar Arvid",0,805,3,male,0,1,347089
592,56.0,A7,C,30.6958,"Smith, Mr. James Clinch",0,175,1,male,0,0,17764
748,NaN,NaN,Q,23.2500,"McCoy, Miss. Agnes",0,331,3,female,2,1,367226
1125,42.0,E24,S,26.2875,"Calderhead, Mr. Edward Pennington",0,708,1,male,0,1,PC 17476
900,50.0,NaN,S,8.0500,"Rouse, Mr. Richard Henry",0,483,3,male,0,0,A/5 3594


In [20]:
fullDF.describe() # quick statistical description of the dataframe

,Age,Fare,Parch,Passengerld,Pclass,SibSP,Survived
count,1046.000000,1308.000000,1309.000000,1309.000000,1309.000000,1309.000000,1309.000000
mean,29.881138,33.295479,0.385027,655.000000,2.294882,0.498854,0.377387
std,14.413493,51.758668,0.865560,378.020061,0.837836,1.041658,0.484918
min,0.170000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,21.000000,7.895800,0.000000,328.000000,2.000000,0.000000,0.000000
50%,28.000000,14.454200,0.000000,655.000000,3.000000,0.000000,0.000000
75%,39.000000,31.275000,0.000000,982.000000,3.000000,1.000000,1.000000
max,80.000000,512.329200,9.000000,1309.000000,3.000000,8.000000,1.000000


In [36]:
def dataMapSex(Sex):

    label = 0

    if Sex == 'male':
        label = 0
    elif Sex == 'female':
        label = 1
    else:
        raise(RuntimeWarning(f'Unknown sex: {Sex}, using default label 0'))
    
    return label

def dataMapAge(Age):
    label = 28

    if Age == 'NaN':
        label = 28
    else:
        raise(RuntimeWarning(f'Unknown Age: {Age}, using default label 0'))
    
    return label

In [37]:
# Let's apply the mapping function to the input data and create a new column called 'Y'
fullDF['SexMap'] = [dataMapSex(item) for item in fullDF['Sex']]
fullDF['Y'] = fullDF['Survived']


In [38]:
# Let's check that mapping works on a few random samples
fullDF.sample(10)

,Age,Cabin,Embarked,Fare,Name,Parch,Passengerld,Pclass,Sex,SibSP,Survived,Ticket,SexMap,Y
1029,NaN,NaN,S,7.0500,"Jardin, Mr. Jose Neto",0,612,3,male,0,0,SOTON/O.Q. 3101305,0,0
159,26.0,NaN,S,13.7750,"Peacock, Mrs. Benjamin (Edith Nile)",2,1051,3,female,0,1,SOTON/O.Q. 3101315,1,1
918,17.0,NaN,S,8.6625,"Calic, Mr. Petar",0,501,3,male,0,0,315086,0,0
1054,32.0,NaN,S,7.9250,"Leinonen, Mr. Antti Gustaf",0,637,3,male,0,0,STON/O 2. 3101292,0,0
232,21.0,NaN,S,6.4958,"Wiklund, Mr. Karl Johan",0,1124,3,male,1,0,3101266,0,0
890,33.0,NaN,S,27.7500,"West, Mrs. Edwy Arthur (Ada Mary Worth)",2,473,2,female,1,1,C.A. 34651,1,1
871,49.0,C92,C,89.1042,"Goldenberg, Mr. Samuel L",0,454,1,male,1,1,17453,0,1
115,18.0,NaN,C,14.4542,"Chronopoulos, Mr. Demetrios",0,1007,3,male,1,0,2680,0,0
23,21.0,NaN,C,61.3792,"Williams, Mr. Richard Norris II",1,915,1,male,0,0,PC 17597,0,0
740,30.0,NaN,Q,12.3500,"Slayter, Miss. Hilda Mary",0,323,2,female,0,1,234818,1,1


In [46]:
dataDF = fullDF[['Age', 'SexMap']]
classDF = fullDF['Y']

## Data Splitting

In [47]:
# Let's split the data into training data, and test data. Same splitting should be applied to classes.
# Here, the test data size is 10% of the full dataset
trainData,testData,trainY,testY = train_test_split(dataDF,classDF,test_size=0.1)

In [48]:
kNN = KNeighborsClassifier(n_neighbors=3,algorithm='kd_tree',metric='minkowski',p=2,n_jobs=-1)
kNN.fit(trainData,trainY)

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values